In [ ]:
import polars as pl

data = pl.read_parquet("../data/1m/1m.parquet").lazy()

In [ ]:
data4 = (
    data.filter(pl.col("symbol") == "BTCUSDT")
    .select("open_time", 
            pl.col("close").alias("BTC_price"),
            pl.col("close").log().alias("BTC"))
    .join(
        data.filter(pl.col("symbol") == "ETHUSDT").select(
            "open_time", 
            pl.col("close").alias("ETH_price"),
            pl.col("close").log().alias("ETH")
        ),
        on="open_time",
    )
).filter((pl.col("open_time")>=pl.datetime(2022,1,3)) & (pl.col("open_time")<pl.datetime(2023,1,1)))

In [ ]:
# Backtest: zscore +/- 2 long-short spread with no lookahead and simple fees

import numpy as np
import polars as pl

# Parameters
entry_z = 2.5
fee_bps = 0.2        # per leg, e.g., 2 bps = 0.02%
fee_rate = fee_bps / 1e4
exit_z = 0.6
window_size = 120

In [ ]:
bt = (
    data4.sort("open_time")
    .with_columns(
        # rolling hedge model
        pl.rolling_cov("BTC", "ETH", window_size=window_size).alias("rolling_cov"),
        pl.col("ETH").rolling_var(window_size=window_size).alias("rolling_var_eth"),
    )
    .with_columns(
        (pl.col("rolling_cov") / pl.col("rolling_var_eth")).alias("rolling_beta"),
    )
    .with_columns(
        (
            pl.col("BTC").rolling_mean(window_size=window_size)
            - pl.col("rolling_beta") * pl.col("ETH").rolling_mean(window_size=window_size)
        ).alias("rolling_alpha"),
    )
    .with_columns(
        (
            pl.col("BTC") - (pl.col("rolling_alpha") + pl.col("rolling_beta") * pl.col("ETH"))
        ).alias("rolling_residual"),
    )
    .with_columns(
        (
            (pl.col("rolling_residual")
             - pl.col("rolling_residual").rolling_mean(window_size=window_size))
            / pl.col("rolling_residual").rolling_std(window_size=window_size)
        ).alias("rolling_zscore"),
    )
    # returns and lagged params to avoid lookahead
    .with_columns(
        pl.col("BTC_price").pct_change().alias("r_btc"),
        pl.col("ETH_price").pct_change().alias("r_eth"),
        pl.col("rolling_beta").shift(1).alias("beta_lag"),
        pl.col("rolling_zscore").shift(1).alias("z_lag"),
    )
    # position from z-lag: -1 short spread when z>+2, +1 long spread when z<-2, 0 otherwise
    .with_columns(
        pl.when(pl.col("z_lag") >= entry_z).then(-1)
         .when(pl.col("z_lag") <= -entry_z).then(1)
         .when(pl.col("z_lag").abs()<= exit_z).then(0)
        .otherwise(None)
        .forward_fill()
        .alias("pos")
    )
    # spread return: r_btc - beta * r_eth, apply position
    .with_columns(
        (pl.col("r_btc") - pl.col("beta_lag") * pl.col("r_eth")).alias("r_spread"),
        pl.col("pos").shift(1).fill_null(0).alias("pos_prev")  # for turnover/fees
    )
    .with_columns(
        (pl.col("pos") * pl.col("r_spread")).alias("pnl_gross"),
        # turnover in spread notional = |Δpos| * (1 + |beta|)
        ((pl.col("pos") - pl.col("pos_prev")).abs() * (1 + pl.col("beta_lag").abs())).alias("turnover_units"))
    .with_columns(
        (fee_rate * pl.col("turnover_units")).alias("fees"))
    .with_columns(
        (pl.col("pnl_gross") - pl.col("fees")).alias("pnl_net"))
    .with_columns(
        (1 + pl.col("pnl_net")).cum_prod().alias("equity"),
        pl.col("pnl_net").cum_sum().alias("cum_return")
    )
    .select(
        "open_time", "rolling_beta", "rolling_zscore", "z_lag",
        "r_btc", "r_eth", "r_spread",
        "pos", "pos_prev", "pnl_gross", "fees", "pnl_net", "equity", "cum_return"
    )
    .drop_nulls()
)

out = bt.collect()

# Basic metrics
ann_factor = 365 * 24 * 60  # minutes per year
mean:float = out["pnl_net"].mean() # type: ignore
std:float = out["pnl_net"].std() # type: ignore
sharpe = (mean / std) * np.sqrt(ann_factor) if std and std > 0 else float("nan")
trades = int((out["pos"] != out["pos_prev"]).sum() // 2)  # entries count (ignores flip as 2 trades)

# Max drawdown on equity
equity = out["equity"].to_numpy()
roll_max = np.maximum.accumulate(equity)
dd = (equity / roll_max) - 1.0
max_dd = dd.min()

print({
    "bars": len(out),
    "trades": trades,
    "mean_minute_pnl": mean,
    "std_minute_pnl": std,
    "sharpe_annualized": sharpe,
    "max_drawdown": float(max_dd),
    "final_equity": float(equity[-1]) if len(equity) else None,
})

# Quick visuals (optional)
# out.plot.line(x="open_time", y="equity")